# Module 6 — Baseline Systems (Standard RAG + LLM-only)

**Goal:** build the two baseline systems the proposal requires for evaluation (Section 7.8):

1. **Standard RAG baseline** — same DeepSeek-r1 model, same FAISS vector index as the KG-RAG system, but no Neo4j knowledge graph pathway and no Symbolic Verifier Agent. This isolates exactly what the graph pathway contributes.
2. **LLM-only baseline** — no retrieval at all. The model answers purely from its own parametric knowledge. This isolates what retrieval (of any kind) contributes.

Both save reports in a structure comparable to Module 5's, so all three systems' outputs for the same query can be compared later.

Unlike Modules 2-5, this notebook needs no scispaCy, no nmslib, and no live Neo4j connection - Standard RAG here only touches the FAISS index and chunk metadata files from Module 1, not the graph at all. (The local version's original note suggesting Neo4j needed to be running for this module was inaccurate and has been dropped.)

Requires Module 1's outputs (FAISS index + chunk metadata) and, for the final comparison step, at least one saved report from Module 5.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install packages

In [2]:
!pip install -q faiss-cpu

In [3]:
!pip install -q transformers ollama

## 3. Install and start Ollama, then pull deepseek-r1:1.5b

Same model as Module 5, kept identical on purpose - a fair baseline comparison depends on both systems using the exact same reasoning model.

In [4]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
import subprocess, time, requests

ollama_process = subprocess.Popen(["ollama", "serve"])

for attempt in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start in time - re-run this cell.")

Ollama server is up.


In [6]:
!ollama pull deepseek-r1:1.5b

## 4. Constants

In [7]:
import os
import re
import gc
import json
import time
from datetime import datetime
import pandas as pd
import numpy as np
import faiss
import torch

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
OLLAMA_MODEL = "deepseek-r1:1.5b"  # SAME model as the KG-RAG system - kept identical for a fair comparison

REPORTS_DIR = f"{PROJECT_ROOT}/reports/baselines"
os.makedirs(REPORTS_DIR, exist_ok=True)

print("Setup ready.")

Setup ready.


## 5. Standard RAG baseline - vector search only, no graph, no verifier

Same memory-safe pattern as Module 5: load, use, free. Pooling is attention-masked mean, matching every other module's corrected version - mismatched pooling between this and the corpus embeddings would make the FAISS distances meaningless.

In [8]:
def run_vector_retrieval_only(query, k=5):
    from transformers import AutoTokenizer, AutoModel

    faiss_index = faiss.read_index(os.path.join(PROCESSED_DIR, "pmc_patients.index"))
    chunks_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "chunks_metadata.parquet"))

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    bert_model.eval()

    inputs = tokenizer([query], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)

    mask = inputs["attention_mask"].unsqueeze(-1)
    summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    query_vec = (summed / counts).cpu().numpy().astype("float32")

    distances, indices = faiss_index.search(query_vec, k)
    matches = []
    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]
        matches.append({"patient_id": str(row["patient_id"]), "rank": rank + 1, "text": row["text"][:400]})

    context = "\n\n".join(f"Patient {m['patient_id']}: {m['text']}" for m in matches)

    del bert_model, tokenizer, faiss_index
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"context": context, "matches": matches}

print("Vector-only retrieval defined.")

Vector-only retrieval defined.


In [9]:
STANDARD_RAG_PROMPT = """You are a clinical reasoning assistant. Using ONLY the patient context below, \
answer the clinician's query. You MUST tag every factual claim exactly like this example:

Example: "The patient shows [CLAIM: elevated respiratory rate] and [CLAIM: low oxygen saturation]."

Patient context:
{context}

Clinician query: {query}

Respond with 2-4 sentences. Every medical fact stated MUST be wrapped in [CLAIM: ...] tags."""

def run_standard_rag(query, k=5):
    import ollama

    start_time = time.time()

    retrieval = run_vector_retrieval_only(query, k=k)

    prompt = STANDARD_RAG_PROMPT.format(context=retrieval["context"] or "No context retrieved.", query=query)
    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    raw_text = response["response"]

    claims = re.findall(r"\[CLAIM:\s*(.*?)\]", raw_text)
    if not claims:
        claims = [s.strip() for s in re.split(r'(?<=[.!?])\s+', raw_text) if len(s.strip()) > 15]

    latency = round(time.time() - start_time, 2)

    return {
        "system": "Standard RAG",
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "recommendation": raw_text,
        "claims": claims,
        "num_claims": len(claims),
        "retrieved_patients": [m["patient_id"] for m in retrieval["matches"]],
        "latency_seconds": latency,
    }

print("Standard RAG function defined.")

Standard RAG function defined.


## 6. LLM-only baseline - no retrieval at all

In [10]:
LLM_ONLY_PROMPT = """You are a clinical reasoning assistant. Answer the clinician's query using your own \
medical knowledge. You MUST tag every factual claim exactly like this example:

Example: "Consider [CLAIM: elevated respiratory rate] and [CLAIM: low oxygen saturation] as warning signs."

Clinician query: {query}

Respond with 2-4 sentences. Every medical fact stated MUST be wrapped in [CLAIM: ...] tags."""

def run_llm_only(query):
    import ollama

    start_time = time.time()

    prompt = LLM_ONLY_PROMPT.format(query=query)
    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    raw_text = response["response"]

    claims = re.findall(r"\[CLAIM:\s*(.*?)\]", raw_text)
    if not claims:
        claims = [s.strip() for s in re.split(r'(?<=[.!?])\s+', raw_text) if len(s.strip()) > 15]

    latency = round(time.time() - start_time, 2)

    return {
        "system": "LLM-only",
        "timestamp": datetime.now().isoformat(),
        "query": query,
        "recommendation": raw_text,
        "claims": claims,
        "num_claims": len(claims),
        "retrieved_patients": [],
        "latency_seconds": latency,
    }

print("LLM-only function defined.")

LLM-only function defined.


## 7. Save results (same structure for both, so they are directly comparable)

In [11]:
def save_baseline_report(report):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    system_tag = report["system"].lower().replace(" ", "_").replace("-", "_")
    json_path = os.path.join(REPORTS_DIR, f"{system_tag}_{ts}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    print(f"Saved: {json_path}")
    return json_path

## 8. Run both baselines on a test query

In [12]:
query = "What should be considered for a patient presenting with COVID-19 and respiratory distress?"

print("=== Running Standard RAG baseline ===")
standard_rag_report = run_standard_rag(query)
save_baseline_report(standard_rag_report)
print(f"Recommendation: {standard_rag_report['recommendation']}\n")
print(f"Claims: {standard_rag_report['num_claims']} | Latency: {standard_rag_report['latency_seconds']}s\n")

print("\n=== Running LLM-only baseline ===")
llm_only_report = run_llm_only(query)
save_baseline_report(llm_only_report)
print(f"Recommendation: {llm_only_report['recommendation']}\n")
print(f"Claims: {llm_only_report['num_claims']} | Latency: {llm_only_report['latency_seconds']}s")

=== Running Standard RAG baseline ===


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved: /content/drive/MyDrive/ClinicalTrust/reports/baselines/standard_rag_20260827_043114.json
Recommendation: [CLAIM: The patient presents with respiratory distress and hypercap acite, which are indicative of a respiratory disorder.]  
[CLAIM: The patient's genetic suspicion indicates a hereditary diffuse gastric carcinoma (HDG) secondary to a genetic mutation in the CDH1 gene, as supported by the genetic deletion of exons 1–2 of CDH1. This genetic fact is an important consideration in the differential diagnosis.]  
[CLAIM: The patient's presentation with respiratory distress may be related to the underlying disease or treatment history, but these factors should not be the sole determinants in clinical practice.]

Claims: 3 | Latency: 196.0s


=== Running LLM-only baseline ===
Saved: /content/drive/MyDrive/ClinicalTrust/reports/baselines/llm_only_20260827_043221.json
Recommendation: [CLAIM: A patient presenting with COVID-19 and respiratory distress should be considered for evaluatio

## 9. Compare all three systems side by side

Loads the most recent Module 5 KG-RAG report from ClinicalTrust/reports/audit_trail/ on Drive - that is where Module 5's Colab version actually saves its reports, not the local version's ../reports path.

In [13]:
def load_latest_report(directory, prefix=""):
    if not os.path.isdir(directory):
        return None
    files = [f for f in os.listdir(directory) if f.startswith(prefix) and f.endswith(".json")]
    if not files:
        return None
    latest = sorted(files)[-1]
    with open(os.path.join(directory, latest), "r", encoding="utf-8") as f:
        return json.load(f)

kg_rag_report = load_latest_report(f"{PROJECT_ROOT}/reports/audit_trail", prefix="audit_report")

comparison_rows = []
if kg_rag_report:
    comparison_rows.append({
        "System": "KG-RAG (proposed)",
        "Num Claims": kg_rag_report.get("total_claims", "N/A"),
        "ATCS": kg_rag_report.get("atcs_score", "N/A"),
        "Verified": kg_rag_report.get("verified_claims", "N/A"),
    })
else:
    print("No Module 5 report found yet in reports/audit_trail - run Module 5 first to include KG-RAG in this comparison.")

comparison_rows.append({
    "System": "Standard RAG",
    "Num Claims": standard_rag_report["num_claims"],
    "ATCS": "N/A (no Verifier)",
    "Verified": "N/A",
})
comparison_rows.append({
    "System": "LLM-only",
    "Num Claims": llm_only_report["num_claims"],
    "ATCS": "N/A (no retrieval)",
    "Verified": "N/A",
})

pd.DataFrame(comparison_rows)

,System,Num Claims,ATCS,Verified
0,KG-RAG (proposed),0,0.0,0
1,Standard RAG,3,N/A (no Verifier),N/A
2,LLM-only,5,N/A (no retrieval),N/A


## Next steps

1. Run the same set of test queries through all three systems (KG-RAG via Module 5, Standard RAG and LLM-only here) - consistency across queries is what the evaluation section needs.
2. Per the proposal (Section 7.8), each configuration needs 3 independent runs with mean/standard deviation reported - wrap the run functions in a loop, e.g. [run_standard_rag(query) for _ in range(3)].
3. Hallucination rate and Clinical F1 require comparing generated claims against the Neo4j graph as ground truth - this can reuse Module 5's verify_claim logic, applied retroactively to baseline outputs too (they just will not have used it during generation).
4. ROUGE-L needs a reference summary per patient to compare against - install rouge-score (pip install rouge-score) when ready for that metric.
5. Once comfortable with a handful of queries, scale to the full evaluation set and compute paired t-tests (scipy.stats.ttest_rel) between KG-RAG and each baseline.
6. Save this notebook into ClinicalTrust/notebooks/ on Drive alongside Modules 1-5.